### Phase1: Build a Schema
Given a JSON document, figure out which fields are searchable and what type they are.

(hardcode now, automate later)

In [ ]:
from datetime import datetime

In [ ]:
def discover_fields(doc):
  fields=[]
  for key in doc:
      fields.append(key)
  return fields

In [ ]:
def determine_type(value):
  if(isinstance(value,int)):
    return "int"
  elif(isinstance(value,float)):
    return "float"
  elif(isinstance(value,bool)):
    return "bool"
  elif(isinstance(value,str)):
    try:
      datetime.strptime(value,"%Y-%m-%d")
      return "date"
    except ValueError:
      return "categorical"

In [ ]:
def build_schema(doc):
  schema={}
  for key,value in doc.items():
    field_type=determine_type(value)
    schema[key] = {
            "type": field_type,
            "indexed": field_type=="categorical"
        }
  return schema

In [ ]:
doc = {
    "profile.firstName": "Alex",
    "profile.lastName": "Chen",
    "profile.dateOfBirth": "1995-08-14",
    "metadata.loginCount": 342,
    "preferences.theme": "dark"
}
schema=build_schema(doc)
print(schema)

### Phase 2: Single ingest

In [ ]:
def index_document(doc,doc_id,schema,index):
  for field,value in doc.items():
    if not schema[field]["indexed"]:
      continue
    if field not in index:
      index[field]={}
    if value not in index[field]:
      index[field][value]=set()
    index[field][value].add(doc_id)

In [ ]:
doc1 = {
    "profile.firstName": "Alex",
    "preferences.theme": "dark",
    "metadata.loginCount": 342
}

doc2 = {
    "profile.firstName": "John",
    "preferences.theme": "dark",
    "metadata.loginCount": 100
}

doc3 = {
    "profile.firstName": "Alex",
    "preferences.theme": "light",
    "metadata.loginCount": 200
}

In [ ]:
schema = build_schema(doc1)
index = {}
index_document(doc1, 0, schema, index)
print(index)

In [ ]:
index_document(doc2, 1, schema, index)

In [ ]:
index_document(doc3, 2, schema, index)
print(index)

#### query engine

In [ ]:
def query_equals(field,value,index):
  if field not in index:
    return set()
  if value not in index[field]:
    return set()
  return index[field][value]

In [ ]:
print(query_equals("profile.firstName","Alex",index))

In [ ]:
# eg, results = [
#     query_equals("profile.firstName", "Alex", index),
#     query_equals("preferences.theme", "dark", index),
#     query_equals("country", "USA", index)
# ]
def query_and(results):
  if not results:
    return set()
  answer=results[0]
  for q in results[1:]:
    answer=answer & q
  return answer

In [ ]:
def query_or(results):
  if not results:
    return set()
  answer=results[0]
  for q in results[1:]:
    answer|=q
  return answer

### Simple BitMap implementation

In [ ]:
class BitMap2:
  def __init__(self):
    self.bits=0
  # Initially,everything absent:0000000000000000

  # Add an element
  def add(self,x):
    self.bits |=(1<<x)
  # to check if an element exists
  def contains(self,x):
    return (self.bits & (1<<x))!=0
  # remove an element
  def remove(self,x):
    self.bits&= ~(1<<x)
  def __str__(self):
    return bin(self.bits)

In [ ]:
bm=BitMap2()
bm.add(2)
bm.add(5)
print(bm)

In [ ]:
bm.add(1)
bm.add(4)
bm.add(7)

In [ ]:
print(bm)

In [ ]:
print(bm.contains(4))

In [ ]:
bm.remove(4)

In [ ]:
print(bm.contains(4))

In [ ]:
print(bm)

#### Next Improvement:
In languages like C++: unsigned long long has only 64 bits, So you can't even store bit 1000.
Instead of one gigantic integer, we'll store many 64-bit integers.

then

word = x // 64

bit = x % 64

In [1]:
class BitMap:
  def __init__(self):
    self.words=[]
  def ensure_capacity(self,word):
    while len(self.words)<=word:
      self.words.append(0)
  def add(self,x):
    word=x//64
    bit=x%64
    self.ensure_capacity(word)
    self.words[word]|=(1<<bit)
  def contains(self,x):
    word=x//64
    bits=x%64
    if(len(self.words)<=word):
      return False
    return self.words[word]&(1<<bits)!=0
  def remove(self,x):
    word=x//64
    bits=x%64
    if(len(self.words)<=word):
      return
    self.words[word]&=~(1<<bits)
  def __and__(self, other):
    result=BitMap()
    for i in range(min(len(self.words),len(other.words))):
      result.words.append(self.words[i]&other.words[i])
    return result
  def __or__(self, other):
    result=BitMap()
    for i in range(max(len(self.words),len(other.words))):
      word1 = self.words[i] if i < len(self.words) else 0
      word2 = other.words[i] if i < len(other.words) else 0
      result.words.append(word1|word2)
    return result
  def __xor__(self, other):
    result=BitMap()
    for i in range(max(len(self.words),len(other.words))):
      word1 = self.words[i] if i < len(self.words) else 0
      word2 = other.words[i] if i < len(other.words) else 0
      result.words.append(word1^word2)
    return result
  def __iter__(self):
    for idx in range(len(self.words)):
      word=self.words[idx]
      while(word>0):
        lowest=word&(-word)
        cnt=0
        # while(lowest):
        #   cnt+=1
        #   lowest>>=1
        # global_bit = idx * 64 + (cnt - 1)
        bit = lowest.bit_length() - 1
        global_bit = idx * 64 + bit
        yield global_bit
        word=word&(word-1)


In [ ]:
bm = BitMap()

bm.add(5)
bm.add(70)

In [ ]:
bm.contains(5)

In [ ]:
bm.contains(6)

In [ ]:
bm.contains(1000)

In [ ]:
bm.add(1000)

In [ ]:
bm.remove(5)

In [ ]:
bm.contains(1000)

In [ ]:
bm.contains(5)

In [2]:
doc0 = "cats are cute"
doc1 = "dogs are cute"
doc2 = "cats love milk"
doc3 = "dogs love bones"

In [4]:
{
    "cats": BitMap(),
    "dogs": BitMap(),
    "cute": BitMap(),
    "love": BitMap()
}

{'cats': <__main__.BitMap at 0x7b15434968a0>,
 'dogs': <__main__.BitMap at 0x7b151ec9c3b0>,
 'cute': <__main__.BitMap at 0x7b151ec9c590>,
 'love': <__main__.BitMap at 0x7b151ec9c5f0>}

### Roaring Bitmap

#### step1: Array Container

In [5]:
from bisect import bisect_left,insort
class ArrayContainer:
  def __init__(self):
    self.values=[]
  #add
  def add(self,x):
    idx=bisect_left(self.values,x)
    if(idx<len(self.values) and self.values[idx]==x):
      return
    insort(self.values,x)
  #remove
  def remove(self,x):
    idx=bisect_left(self.values,x)
    if(idx<len(self.values) and self.values[idx]==x):
      del self.values[idx]
  #contains
  def contains(self,x):
    idx=bisect_left(self.values,x)
    if(idx<len(self.values) and self.values[idx]==x):
      return True
    return False
  #iter
  def __iter__(self):
    for value in self.values:
      yield value

In [ ]:
class RoaringBitmap: